# ⚡ Inference & Optimization Lab
### Tasar'u · NVIDIA Platform & Cert Prep — hands-on lab

You *serve* a model, you don't run a raw checkpoint. This lab walks the real path from the
deck: **baseline → quantize → serve with a real engine → benchmark.**

| Step | What you do | Maps to (slides) |
|---|---|---|
| 1 | Baseline 🤗 generation, measure tok/s & memory | the starting point |
| 2 | **4-bit (NF4)** quantization → fit a **7B on one T4** | TensorRT / quantization |
| 3 | Batching & KV-cache effects | in-flight batching / paged KV |
| 4 | Serve with **vLLM** (continuous batching) | Triton / NIM |
| 5 | Benchmark table + reflection | AI Operations |

**Platform:** Colab or Kaggle, a single **T4 (16 GB)** is enough.


## 0 · Setup

In [ ]:
!pip -q install "transformers>=4.44" accelerate bitsandbytes matplotlib 2>/dev/null
import torch, time
def stats(): return torch.cuda.max_memory_allocated()/1e9
print("torch", torch.__version__, "| GPU", torch.cuda.get_device_name(0))

## 1 · Baseline generation
Load a small chat model in fp16 and measure two things every inference team cares about:
**throughput** (tokens/second) and **peak memory**.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
BASE = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16).cuda().eval()

prompt = "Explain what GPU utilization means in one sentence."
def bench(model, tok, prompt, new=64, runs=3):
    ids = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad(): model.generate(**ids, max_new_tokens=8)   # warmup
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    t0=time.time()
    for _ in range(runs):
        with torch.no_grad(): out = model.generate(**ids, max_new_tokens=new, do_sample=False)
    torch.cuda.synchronize(); dt=(time.time()-t0)/runs
    return new/dt, torch.cuda.max_memory_allocated()/1e9

tps, mem = bench(model, tok, prompt)
print(f"Baseline fp16: {tps:.1f} tok/s | peak {mem:.2f} GB")

## 2 · 4-bit quantization → fit a 7B on a 16 GB T4
A 7B model in fp16 is ~14 GB and barely fits (if at all) with activations + KV cache.
Load it in **4-bit NF4** instead: weights shrink ~4×, so a 7B lands around ~5–6 GB —
comfortable on a T4. This is the same "make it smaller to make it deployable" idea as
**TensorRT / TensorRT-LLM** quantization.

In [ ]:
from transformers import BitsAndBytesConfig
del model; torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

BIG = "Qwen/Qwen2.5-7B-Instruct"   # open model, no gating
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tok7 = AutoTokenizer.from_pretrained(BIG)
model7 = AutoModelForCausalLM.from_pretrained(BIG, quantization_config=bnb, device_map="auto")
print("Loaded 7B in 4-bit. Weight memory footprint:", f"{torch.cuda.memory_allocated()/1e9:.2f} GB")

tps7, mem7 = bench(model7, tok7, prompt)
print(f"7B @ 4-bit: {tps7:.1f} tok/s | peak {mem7:.2f} GB  (a 7B that would NOT fit in fp16 on a T4)")

## 3 · Batching and the KV cache
**Batching** amortizes GPU work across requests — the single biggest lever for serving
throughput. And the **KV cache** (`use_cache=True`) stores past attention keys/values so
each new token is cheap. Let's see both.

In [ ]:
# Throughput vs batch size on the small model (reload it)
model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16).cuda().eval()
prompts = ["Describe an AI data center." ] * 16
res=[]
for B in [1, 4, 8, 16]:
    batch = tok(prompts[:B], return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad(): model.generate(**batch, max_new_tokens=8)  # warmup
    torch.cuda.synchronize(); t0=time.time()
    with torch.no_grad(): out = model.generate(**batch, max_new_tokens=48, do_sample=False)
    torch.cuda.synchronize(); dt=time.time()-t0
    tput = B*48/dt; res.append((B, tput)); print(f"batch {B:2d}: {tput:6.1f} tok/s aggregate")
print("\n→ Same GPU, ~linear throughput gain from batching. This is why servers batch requests.")

In [ ]:
# KV cache on vs off — cost per token
import time
ids = tok("The role of the KV cache is", return_tensors="pt").to(model.device)
for use_cache in [True, False]:
    with torch.no_grad(): model.generate(**ids, max_new_tokens=8, use_cache=use_cache)  # warmup
    torch.cuda.synchronize(); t0=time.time()
    with torch.no_grad(): model.generate(**ids, max_new_tokens=64, do_sample=False, use_cache=use_cache)
    torch.cuda.synchronize(); print(f"use_cache={use_cache}: {(time.time()-t0)*1000:.0f} ms for 64 tokens")
print("\n→ Without the KV cache, every step recomputes all past tokens — much slower.")

## 4 · Serve it with a real engine (vLLM)
🤗 `generate()` runs one request at a time. Production servers like **vLLM** (and Triton /
NIM) do **continuous batching** — new requests slot into the running batch — for far higher
throughput under load. Here we use vLLM's offline engine on a small model.

> ⚠️ vLLM support on T4 (Turing) is version-sensitive. If the install or load fails, that's OK —
> read the concept and move on; the exam point is *what a serving engine does*, not the install.

In [ ]:
try:
    !pip -q install vllm 2>/dev/null
    from vllm import LLM, SamplingParams
    llm = LLM(model="TinyLlama/TinyLlama-1.1B-Chat-v1.0", dtype="float16", gpu_memory_utilization=0.6, max_model_len=1024)
    sp = SamplingParams(max_tokens=48, temperature=0)
    many = ["Explain " + t for t in ["MIG","NVLink","InfiniBand","GPUDirect","vLLM","batching","the KV cache","quantization"]]
    import time; t0=time.time()
    outs = llm.generate(many, sp)          # all 8 served together via continuous batching
    dt=time.time()-t0
    total = sum(len(o.outputs[0].token_ids) for o in outs)
    print(f"vLLM served {len(many)} requests, {total} tokens in {dt:.1f}s = {total/dt:.0f} tok/s aggregate")
except Exception as e:
    print("vLLM unavailable on this runtime — skipping the live run.")
    print("Concept: a serving engine keeps the GPU saturated by merging requests into one rolling batch.")
    print("Detail:", str(e)[:160])

## 5 · Benchmark summary & reflection

| Configuration | Fits on T4? | Throughput | Memory |
|---|---|---|---|
| 1.1B fp16 (baseline) | ✅ | (your §1 number) | ~2–3 GB |
| **7B @ 4-bit NF4** | ✅ | (your §2 number) | ~5–6 GB |
| 1.1B + batching | ✅ | scales with batch | one GPU |
| 1.1B on vLLM | ✅ | highest under load | one GPU |

**Reflection (write your answers):**
1. 4-bit let a 7B fit on a T4. What did you likely trade away? How would you *measure* that
   loss? *(you will — in the QLoRA + Eval lab)*
2. Batching raised throughput but what does it do to a single request's **latency**? Why do
   real servers cap batch size / wait time?
3. Name the three tools from the slides for: making inference *fast*, *serving* it, and
   shipping a *prebuilt* model API. *(TensorRT · Triton · NIM)*
